
# 청킹 사람 검수 (Chunking Review)

이 노트북은 **채점하는 곳이 아니라 눈으로 보는 곳**이다.
`chunking_evaluation.ipynb`가 재는 것은 "규칙을 지켰는가"뿐이고,
**"읽을 만한가 · 인용이 맞는가 · 잘린 자리가 어색한가"는 사람만 판단할 수 있다**
(전략 §10.4, 결정대기 ⑥).

## 무엇을 보는가 — 5개 검수 큐

| 큐 | 대상 | 왜 사람이 봐야 하는가 |
|---|---|---|
| **A** 의심 청크 | 거짓 인용 · 플래그 · 예산 초과 · 파편 | 자동 판정이 **의심만** 하고 옳고 그름은 못 정한다 |
| **B** 표 | 논리 표 39개 | 한도표가 **룰 임계값의 원천**이다. 값이 깨지면 룰이 조용히 틀린다 |
| **C** 쪼개진 조 | 부모 89개 | "잘린 자리가 어색한가"는 지표가 아니라 감각이다 |
| **D** 인용 표기 | 규정의 `1. 2. 3.` | **항인가 호인가** — §11 ④ **확정**(조 단위 판정). 이제 사람이 정하는 곳이 아니라 **구현이 원문과 맞는지 확인하는 곳**이다 |
| **E** 표본 40건 | 버킷 할당 추출 | 의심 없는 청크도 봐야 편향이 안 생긴다 |

## 검수 기록 방법

셀을 실행해 청크를 보고, 아래처럼 판정을 남긴다. **재실행해도 기록은 남는다**
(`output/chunking/review_filled.csv`에서 다시 읽어 온다).

```python
mark("dump:회식_사용규정#c01a002#03", q1="N", q2="N", memo="제6조 문장이 섞임")
save_review()      # 저장 — 자주 눌러도 된다
progress()         # 진행률
```

- **Q1** 이 청크만으로 뜻이 통하는가 (Y/N)
- **Q2** 인용(조·항)이 본문과 일치하는가 (Y/N)
- **Q3** 잘린 곳이 어색한가 (**Y가 나쁨**)


---
## 1. 준비 — 덤프 → 교정 → 청킹

평가 노트북과 **같은 경로**다(파싱 덤프 → C1~C7 → 청커). docling을 다시 돌리지 않으므로 수초면 끝난다.
여기서 보는 청크가 곧 운영에서 나오는 청크다.

In [1]:
from __future__ import annotations

import csv
import html as _html
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 220)


def _resolve_base_dir() -> Path:
    """CWD가 레포 루트든 docling_eval 내부든 같은 docling_eval 을 가리키게 한다."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if cand.name == "docling_eval":
            return cand
        if (cand / "docling_eval").is_dir() or (cand / "tiger_inc").is_dir():
            return cand / "docling_eval"
    return cwd / "docling_eval"


BASE_DIR = _resolve_base_dir()
REPO_ROOT = BASE_DIR.parent
AI_APP = REPO_ROOT / "apps" / "ai"
if str(AI_APP) not in sys.path:
    sys.path.insert(0, str(AI_APP))          # 청킹 구현은 앱 코드가 정본이다

OUTPUT_DIR = BASE_DIR / "output"
LAYOUT_CSV = OUTPUT_DIR / "layout" / "layout_result.csv"
TABLES_DIR = OUTPUT_DIR / "tables"
CHUNK_DIR = OUTPUT_DIR / "chunking"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)

# ▼▼▼ 바꿀 곳은 여기뿐 ▼▼▼
TARGET_DOCS = None            # None = 전체. 일부만 볼 때: ["법인카드_사용규정"]
BUDGET_OVERRIDE = None        # 예산을 바꿔 보고 싶을 때: dict(target=500, max=800)
REVIEW_PATH = CHUNK_DIR / "review_filled.csv"     # 검수 기록 (재실행해도 이어서)
SAMPLE_SHEET = CHUNK_DIR / "review_samples.csv"   # 평가 노트북이 뽑아 둔 40건
# ▲▲▲

if not LAYOUT_CSV.exists():
    raise FileNotFoundError(
        f"파싱 덤프가 없습니다: {LAYOUT_CSV}\n"
        "→ 먼저 docling_eval/docling_parsing_test.ipynb 를 끝까지 실행하세요."
    )

from app.rag.chunking import chunk_document                  # noqa: E402
from app.rag.chunking.model import Budget                    # noqa: E402
from app.rag.parsing import dump as dump_mod                 # noqa: E402
from app.rag.parsing.corrections import pipeline             # noqa: E402

BUDGET = Budget(**BUDGET_OVERRIDE) if BUDGET_OVERRIDE else Budget()

print(f"BASE_DIR  : {BASE_DIR}")
print(f"청킹 구현 : {AI_APP / 'app' / 'rag' / 'chunking'}")
print(f"예산      : target={BUDGET.target} max={BUDGET.max} hard={BUDGET.hard}")

BASE_DIR  : D:\project\SKN29-FINAL-1TEAM\docling_eval
청킹 구현 : D:\project\SKN29-FINAL-1TEAM\apps\ai\app\rag\chunking
예산      : target=800 max=1200 hard=2000


In [2]:
DOCS: dict = {}          # {문서명: ParsedDoc}
CHUNKS: dict = {}        # {문서명: list[Chunk]}
REPORTS: dict = {}       # {문서명: ChunkReport}

for name, doc in dump_mod.load_all(LAYOUT_CSV, TABLES_DIR).items():
    if TARGET_DOCS and name not in TARGET_DOCS:
        continue
    pipeline.run(doc)                       # 교정 — 청킹은 교정된 계층에만 의존한다
    chunks, report = chunk_document(doc, BUDGET)
    DOCS[name], CHUNKS[name], REPORTS[name] = doc, chunks, report

ALL_CHUNKS = [c for cs in CHUNKS.values() for c in cs]
LEAVES = [c for c in ALL_CHUNKS if c.chunk_role != "parent"]
BY_ID = {c.chunk_id: c for c in ALL_CHUNKS}

# 요소 → 조 번호 (거짓 인용 판정에 쓴다)
ART_OF: dict[str, dict] = {
    name: {e.element_id: e.attrs.get("article_no") for e in doc.elements}
    for name, doc in DOCS.items()
}

print(f"문서 {len(DOCS)}종 · 청크 {len(ALL_CHUNKS):,}개 "
      f"(잎 {len(LEAVES):,} / 부모 {len(ALL_CHUNKS) - len(LEAVES)})")
for name in DOCS:
    print(f"  {name:<20} [{DOCS[name].profile:<10}] {len(CHUNKS[name]):>4}청크")

문서 11종 · 청크 888개 (잎 799 / 부모 89)
  법인카드_사용규정            [REGULATION]   25청크
  부서소개                 [DIAGRAM   ]    6청크
  업무추진비_사용규정           [REGULATION]   23청크
  조직도                  [DIAGRAM   ]   10청크
  조직설계_상세기획서           [DIAGRAM   ]   30청크
  직급체계                 [DIAGRAM   ]    9청크
  출장비_사용규정             [REGULATION]   24청크
  회식_운영규정              [REGULATION]   31청크
  법인세법                 [LAW       ]  425청크
  부가가치세법               [LAW       ]  130청크
  여신전문금융업법             [LAW       ]  175청크



---
## 2. 읽기 도구

청크를 **저장된 그대로**(헤더 + 본문) 보여 준다. `context_text()`가 LLM에 실제로 들어가는 문자열이므로,
여기서 읽기 나쁘면 LLM도 읽기 나쁘다.

| 함수 | 용도 |
|---|---|
| `show(c)` | 청크 하나 (문자열 ID도 됨) |
| `show_all(리스트)` | 여러 개 연속 |
| `find("사전승인")` | 본문 키워드 검색 — **질의 정답셋 30건 만들 때 쓴다** |
| `article_view("회식_사용규정", 6)` | 한 조를 청크 경계(✂)와 함께 통째로 |
| `around(c)` | 앞뒤 이웃 청크까지 (검색 시 확장되는 범위) |

In [3]:
_BOX = ("border:1px solid #d0d7de;border-radius:8px;padding:12px 16px;margin:12px 0;"
        "background:#f6f8fa;color:#1f2328;font-size:13px;line-height:1.7")
_MONO = "font-family:ui-monospace,SFMono-Regular,Consolas,monospace;font-size:11px;color:#57606a"
_BADGE = ("display:inline-block;padding:1px 7px;border-radius:10px;font-size:11px;"
          "margin-left:6px;background:#ffe8cc;color:#7a4b00")


def _esc(s: str) -> str:
    return _html.escape(s or "")


def _card(c, note: str = "", body_override: str | None = None) -> str:
    flags = "".join(f'<span style="{_BADGE}">{_esc(f)}</span>' for f in c.flags)
    if c.chunk_id in REVIEW:
        r = REVIEW[c.chunk_id]
        seen = (f'<span style="{_BADGE};background:#d4f4dd;color:#0a5c2e">검수함 '
                f'Q1={r.get("q1") or "-"} Q2={r.get("q2") or "-"} Q3={r.get("q3") or "-"}</span>')
    else:
        seen = ""
    head = _esc(c.header).replace("\n", "<br>")
    body = _esc(body_override if body_override is not None else c.text)
    note_html = f'<div style="color:#9a6700;margin-bottom:6px">⚠ {_esc(note)}</div>' if note else ""
    return f"""<div style="{_BOX}">
  {note_html}
  <div style="{_MONO}">{_esc(c.chunk_id)} · {c.chunk_type}/{c.chunk_role} · {c.size}자 ·
    p{c.page_start}-{c.page_end} · 요소 {len(c.element_ids)}개{flags}{seen}</div>
  <div style="margin:8px 0 4px;color:#0969da">{head or "<i>헤더 없음</i>"}</div>
  <pre style="white-space:pre-wrap;word-break:break-word;margin:6px 0 0;font-family:inherit"
    >{body}</pre>
</div>"""


def show(c, note: str = "") -> None:
    """청크 하나를 저장된 모습 그대로 렌더한다."""
    if isinstance(c, str):
        c = BY_ID[c]
    display(HTML(_card(c, note)))


def show_all(chunks, limit: int = 20, notes: dict | None = None) -> None:
    chunks = list(chunks)
    for c in chunks[:limit]:
        show(c, (notes or {}).get(c.chunk_id, ""))
    if len(chunks) > limit:
        print(f"… {len(chunks) - limit}건 더 있음 (limit 인자를 올리세요)")


def find(keyword: str, limit: int = 8, docs=None, leaves_only: bool = True):
    """본문 키워드 검색. 임베딩이 없으므로 문자열 매칭이다 — 질의셋 후보 찾기용."""
    pool = LEAVES if leaves_only else ALL_CHUNKS
    hits = [c for c in pool
            if keyword in c.context_text() and (not docs or c.doc_name in docs)]
    print(f'"{keyword}" — {len(hits)}건')
    show_all(hits, limit)
    return hits


def article_view(doc_name: str, article_no: int, limit: int = 8,
                 with_parent: bool = False) -> None:
    """한 조를 이루는 청크를 문서 순서대로, 잘린 자리에 ✂ 를 넣어 보여 준다."""
    cs = [c for c in CHUNKS[doc_name] if c.article_no == article_no]
    if not cs:
        print(f"{doc_name} 제{article_no}조 — 청크 없음")
        return
    parents = [c for c in cs if c.chunk_role == "parent"]
    kids = [c for c in cs if c.chunk_role != "parent"]
    print(f"■ {doc_name} 제{article_no}조 — 청크 {len(kids)}개"
          f"{' (+부모 1)' if parents else ''} · 합계 {sum(k.size for k in kids)}자")
    for i, c in enumerate(kids[:limit]):
        if i:
            display(HTML('<div style="text-align:center;color:#cf222e;font-size:18px">✂</div>'))
        show(c)
    if len(kids) > limit:
        print(f"… {len(kids) - limit}건 더 있음 (limit 인자를 올리세요)")
    if with_parent:
        for p in parents:
            show(p, note="부모 청크 — 검색 대상이 아니라 확장용(§6)")


def around(c, n: int = 1) -> None:
    """이웃 확장(prev/next)으로 붙는 범위를 눈으로 확인한다 — 오버랩 대신 쓰는 장치(§7)."""
    if isinstance(c, str):
        c = BY_ID[c]
    seq, cur = [c], c
    for _ in range(n):
        if cur.prev_chunk_id and cur.prev_chunk_id in BY_ID:
            cur = BY_ID[cur.prev_chunk_id]
            seq.insert(0, cur)
    cur = c
    for _ in range(n):
        if cur.next_chunk_id and cur.next_chunk_id in BY_ID:
            cur = BY_ID[cur.next_chunk_id]
            seq.append(cur)
    for x in seq:
        show(x, note="◀ 기준 청크" if x is c else "")


print("도구 준비 완료 — show / show_all / find / article_view / around")

도구 준비 완료 — show / show_all / find / article_view / around



---
## 3. 검수 기록 — `mark()` · `save_review()`

기록은 **파일이 진실**이다. 노트북을 껐다 켜도 `review_filled.csv`에서 다시 읽어 이어서 검수한다.
자동 판정과 사람 판정이 **어긋난 건**을 나중에 찾을 수 있게, 청크의 플래그·인용도 같이 저장한다.

In [4]:
REVIEW: dict[str, dict] = {}

def _load_review() -> None:
    REVIEW.clear()
    if REVIEW_PATH.exists():
        with REVIEW_PATH.open(encoding="utf-8-sig", newline="") as fh:
            for row in csv.DictReader(fh):
                REVIEW[row["chunk_id"]] = {
                    "q1": row.get("q1", ""), "q2": row.get("q2", ""),
                    "q3": row.get("q3", ""), "memo": row.get("memo", ""),
                    "queue": row.get("queue", ""),
                }
    print(f"기존 검수 기록 {len(REVIEW)}건 로드 ({REVIEW_PATH.name})")


def mark(chunk_id: str, q1: str = "", q2: str = "", q3: str = "",
         memo: str = "", queue: str = "") -> None:
    """판정 기록. 이미 있으면 준 값만 덮어쓴다(부분 수정 가능)."""
    if chunk_id not in BY_ID:
        raise KeyError(f"모르는 청크 ID: {chunk_id}")
    cur = REVIEW.setdefault(chunk_id, {"q1": "", "q2": "", "q3": "", "memo": "", "queue": ""})
    for key, val in (("q1", q1), ("q2", q2), ("q3", q3), ("memo", memo), ("queue", queue)):
        if val:
            cur[key] = val.upper() if key.startswith("q") else val
    c = BY_ID[chunk_id]
    print(f"기록 · {c.citation or c.chunk_id} → "
          f"Q1={cur['q1'] or '-'} Q2={cur['q2'] or '-'} Q3={cur['q3'] or '-'} {cur['memo']}")


def save_review() -> None:
    rows = []
    for cid, r in REVIEW.items():
        c = BY_ID.get(cid)
        rows.append({
            "chunk_id": cid,
            "queue": r.get("queue", ""),
            "document": c.doc_name if c else "",
            "citation": c.citation if c else "",
            "type": f"{c.chunk_type}/{c.chunk_role}" if c else "",
            "size": c.size if c else "",
            "flags": ",".join(c.flags) if c else "",
            "q1": r.get("q1", ""), "q2": r.get("q2", ""), "q3": r.get("q3", ""),
            "memo": r.get("memo", ""),
        })
    pd.DataFrame(rows).to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")
    print(f"저장 {len(rows)}건 → {REVIEW_PATH}")


def progress() -> None:
    done = [r for r in REVIEW.values() if r.get("q1") or r.get("q2") or r.get("q3")]
    bad1 = [k for k, r in REVIEW.items() if r.get("q1") == "N"]
    bad2 = [k for k, r in REVIEW.items() if r.get("q2") == "N"]
    bad3 = [k for k, r in REVIEW.items() if r.get("q3") == "Y"]
    print(f"검수 {len(done)}건 · 큐별 {pd.Series([r.get('queue') or '?' for r in REVIEW.values()]).value_counts().to_dict()}")
    print(f"  Q1 뜻 안 통함 {len(bad1)} · Q2 인용 불일치 {len(bad2)} · Q3 어색한 절단 {len(bad3)}")
    for label, ids in (("Q1=N", bad1), ("Q2=N", bad2), ("Q3=Y", bad3)):
        for cid in ids[:5]:
            print(f"    [{label}] {cid} — {REVIEW[cid].get('memo', '')}")


_load_review()

기존 검수 기록 0건 로드 (review_filled.csv)



---
## 4. 큐 A — 자동 판정이 의심한 것 (우선순위 최상)

자동 지표는 **의심까지만** 한다. 아래 넷은 "이게 실제로 문제인가"를 사람이 정해야 하는 것들이다.


### A-1. 거짓 인용 · 조 경계 위반

한 청크가 **두 조의 요소**를 담은 경우다. 그중 청크의 인용과 어긋나는 것이 **거짓 인용**이고,
이건 "제N조 위반"이라고 회계 담당자에게 잘못 말하는 것이라 가장 위험하다.

**볼 것**: 본문을 읽고 ⓐ 정말 남의 조문이 섞였는지 ⓑ 아니면 파싱(C2)이 조 번호를 잘못 붙인 것인지.
후자면 청킹이 아니라 파싱으로 이관한다(전략 §11 ⑦).

In [5]:
cross_rows, cross_chunks, notes = [], [], {}
for name, chunks in CHUNKS.items():
    for c in (x for x in chunks if x.chunk_role != "parent"):
        arts = {ART_OF[name].get(eid) for eid in c.element_ids} - {None}
        if len(arts) <= 1:
            continue
        wrong = c.article_no is not None and bool(arts - {c.article_no})
        cross_rows.append({"Document": name, "Chunk ID": c.chunk_id, "Citation": c.citation,
                           "Articles": sorted(arts), "거짓 인용": wrong})
        cross_chunks.append(c)
        notes[c.chunk_id] = (f"인용 '{c.citation}' vs 소유 요소 조번호 {sorted(arts)}"
                             + (" — 거짓 인용" if wrong else " — 인용과는 안 어긋남"))

print(f"조 경계 위반(raw) {len(cross_rows)}건 · 그중 거짓 인용 "
      f"{sum(r['거짓 인용'] for r in cross_rows)}건")
if cross_rows:
    display(pd.DataFrame(cross_rows))
    show_all(cross_chunks, notes=notes)
else:
    print("없음")

# 판정 예시:
# mark("dump:회식_사용규정#c01a002#03", q2="N", memo="제6조 문장 혼입 — 파싱 C2 태깅 잔재", queue="A1")

조 경계 위반(raw) 2건 · 그중 거짓 인용 1건


,Document,Chunk ID,Citation,Articles,거짓 인용
0,회식_운영규정,dump:회식_운영규정#c01a002#03,회식_운영규정 제2조,"[2, 6]",True
1,법인세법,dump:법인세법#s002#01,법인세법,"[55, 75]",False



### A-2. 플래그 — 지우지 않고 표시만 한 것

`toc_like`(목차 잔재) · `marker_uncertain`(파싱이 항 번호를 확신 못 함).
**삭제 판단은 인덱싱 정책의 몫**이므로(전략 §9), 여기서 "인덱스에 넣을 가치가 있는가"를 정한다.

- 목차 잔재인데 인용까지 달려 있으면 → 인덱싱 제외 후보
- 항 번호가 틀렸으면 → 인용 신뢰도 하향 대상

In [6]:
flagged = [c for c in ALL_CHUNKS if {"toc_like", "marker_uncertain"} & set(c.flags)]
print(f"플래그 청크 {len(flagged)}건 — "
      f"{pd.Series([f for c in flagged for f in c.flags]).value_counts().to_dict()}")
display(pd.DataFrame([{
    "Document": c.doc_name, "Chunk ID": c.chunk_id, "Flags": ",".join(c.flags),
    "Citation": c.citation, "Size": c.size,
    "Text": c.text[:60].replace("\n", " ⏎ "),
} for c in flagged]))

show_all(flagged, limit=6)     # limit 올려서 전수 확인 가능

플래그 청크 19건 — {'marker_uncertain': 18, 'toc_like': 1}


,Document,Chunk ID,Flags,Citation,Size,Text
0,법인세법,dump:법인세법#s002#01,toc_like,법인세법,1125,제55조의2(토지등 양도소득에 대한 과세특례) ⏎ 제56조 제57조(외국 납부 세액공제 등) 제57조의2(간접투
1,법인세법,dump:법인세법#c01a059#01,marker_uncertain,법인세법 제46조의3 제2항,369,제46조의3(적격분할 시 분할신설법인등에 대한 과세특례) ① 적격분할을 한 분할신설법인등은 제46조의2에도
2,법인세법,dump:법인세법#c01a059#02,marker_uncertain,법인세법 제46조의3 제3~6항,1345,③ 적격분할을 한 분할신설법인등은 3년 이내의 범위에서 대통령령으로 정하는 기간에 다음 각 호의 어느 하나에
3,법인세법,dump:법인세법#c01a060#01,marker_uncertain,법인세법 제46조의4 제2~3항,846,제46조의4(분할 시 이월결손금 등 공제 제한) ① 분할합병의 상대방법인의 분할등기일 현재 제13조제1항제1
4,법인세법,dump:법인세법#c01a060#02,marker_uncertain,법인세법 제46조의4 제4~5항,833,④ 제46조의3제2항에 따라 분할신설법인등이 승계한 분할법인등의 감면 또는 세액공제는 분할법인등으로부터 승계
5,법인세법,dump:법인세법#c01a060#03,marker_uncertain,법인세법 제46조의4 제7~8항,408,⑦ 분할법인등의 분할등기일 현재 기부금한도초과액으로서 제46조의3제2항에 따라 분할신설법인등이 승계한 금액은
6,법인세법,dump:법인세법#c01a089#01,marker_uncertain,법인세법 제62조의2 제2~4항,1078,제62조의2(비영리내국법인의 자산양도소득에 대한 신고 특례) ① 비영리내국법인(제4조제3항제1호에 따른 수익
7,법인세법,dump:법인세법#c01a089#02,marker_uncertain,법인세법 제62조의2 제5~9항,674,"⑤ 자산양도소득에 대한 과세표준의 계산에 관하여는 「소득세법」 제101조 및 제102조를 준용하고, 자산양도"
8,법인세법,dump:법인세법#c04a167#02,marker_uncertain,법인세법 제98조 제2~7항,827,② 삭제<2022. 12. 31.> ⏎ ③ 삭제<2011. 12. 31.> ⏎ ④ 납세지 관할 세무서장은 원천징수의
9,법인세법,dump:법인세법#c04a167#03,marker_uncertain,법인세법 제98조 제8~12항,817,"⑧ 외국법인에 건축, 건설, 기계장치 등의 설치ᆞ조립, 그 밖의 작업이나 그 작업의 지휘ᆞ감독 등에 관한 용"


… 13건 더 있음 (limit 인자를 올리세요)



### A-3. 예산 초과 14건

`max`(1,200자)를 넘었지만 `hard`(2,000자)에는 안 걸린 청크다. 항 하나가 통째로 큰 경우라
쪼갤 자리가 없다. **볼 것**: 정말 더 못 쪼개는지, 아니면 안 쪼갠 것인지.

In [7]:
over = sorted([c for c in LEAVES if c.size > BUDGET.max], key=lambda c: -c.size)
print(f"예산({BUDGET.max}자) 초과 {len(over)}건 · "
      f"{pd.Series([c.doc_name for c in over]).value_counts().to_dict()}")
display(pd.DataFrame([{
    "Document": c.doc_name, "Chunk ID": c.chunk_id, "Size": c.size,
    "Citation": c.citation, "Type": f"{c.chunk_type}/{c.chunk_role}",
} for c in over]))

show_all(over, limit=3)

예산(1200자) 초과 14건 · {'법인세법': 12, '조직설계_상세기획서': 1, '부가가치세법': 1}


,Document,Chunk ID,Size,Citation,Type
0,법인세법,dump:법인세법#c01a077#02,1873,법인세법 제55조의2 제2항,clause/child
1,법인세법,dump:법인세법#c01a030#02,1845,법인세법 제24조 제2항,clause/child
2,법인세법,dump:법인세법#c04a168#01,1722,법인세법 제98조,clause/child
3,법인세법,dump:법인세법#c01a113#01,1683,법인세법 제75조의8 제2~3항,article/atomic
4,조직설계_상세기획서,dump:조직설계_상세기획서#s011#01,1655,조직설계_상세기획서,table/atomic
5,법인세법,dump:법인세법#c01a002#01,1599,법인세법 제2조,article/atomic
6,부가가치세법,dump:부가가치세법#c03a027#01,1469,부가가치세법 제26조 제2항,article/atomic
7,법인세법,dump:법인세법#c04a167#01,1402,법인세법 제98조,clause/child
8,법인세법,dump:법인세법#c02a131#02,1363,법인세법 제76조의14 제2~3항,clause/child
9,법인세법,dump:법인세법#c01a059#02,1345,법인세법 제46조의3 제3~6항,clause/child


… 11건 더 있음 (limit 인자를 올리세요)



### A-4. 파편 91건 — 유일한 실질 감점

100자 미만이면서 **분할로 생긴 자식**인 청크다(Fragment-Free 0.886).
예산을 바꿔도 91에서 안 움직이므로 **예산 문제가 아니라 코퍼스 성질**이라는 게 자동 판정의 결론인데,
그게 맞는지 눈으로 확인할 자리다.

**볼 것**: 이 조각이 ⓐ `② 삭제 <2020. 3. 24.>` 처럼 **원래 짧은 항**인가
ⓑ 아니면 앞 청크에 붙였어야 할 **꼬리**인가. ⓑ가 많으면 `min_merge`(250자)를 올릴 근거가 된다.

In [8]:
frags = [c for c in LEAVES if c.chunk_role == "child" and c.size < 100]
print(f"파편 {len(frags)}건 / 잎 {len(LEAVES)} = {len(frags)/len(LEAVES):.1%}")
display(pd.Series([c.doc_name for c in frags]).value_counts().rename("파편").to_frame())
display(pd.DataFrame([{
    "Chunk ID": c.chunk_id, "Size": c.size, "Citation": c.citation,
    "Text": c.text[:70].replace("\n", " ⏎ "),
} for c in frags]).head(40))

# 파편은 "앞 청크와 이어서" 봐야 판단이 된다 — around() 로 문맥째 확인
for c in frags[:3]:
    print("=" * 90)
    around(c, n=1)

파편 91건 / 잎 799 = 11.4%


,파편
법인세법,53
여신전문금융업법,26
회식_운영규정,5
조직도,2
직급체계,2
법인카드_사용규정,1
조직설계_상세기획서,1
출장비_사용규정,1


,Chunk ID,Size,Citation,Text
0,dump:법인카드_사용규정#s002#01,72,법인카드_사용규정,| 제정일 | 2026. 7. 20. | ⏎ |---|---| ⏎ | 시행일 | 2026. 8. 1. | ⏎ | 소관부서 | 경영지원본부
1,dump:조직도#s002#01,15,조직도,내부참고문서 ⏎ 1. 회사 개요
2,dump:조직도#s002#03,87,조직도,"중견기업 판정 근거: 정보통신업 중소기업 매출액 기준(약 800억원 이하)을 연 매출 약 1,200억원이 초과하여 「중소기업기"
3,dump:조직설계_상세기획서#s022#01,46,조직설계_상세기획서,"금액 기준은 이 표에서 다루지 않으며, 「법인카드 사용 규정」 별표1에서 관리한다."
4,dump:직급체계#s004#02,23,직급체계,총 10단계 (사원 ~ 대표이사)로 구성.
5,dump:직급체계#s005#01,46,직급체계,"금액 기준은 이 표에서 다루지 않으며, 「법인카드 사용 규정」 별표1에서 관리한다."
6,dump:출장비_사용규정#s002#01,80,출장비_사용규정,| 제정일 | 2026. 7. 20. | ⏎ |---|---| ⏎ | 시행일 | 2026. 8. 1. | ⏎ | 소관부서 | 경영지원본부
7,dump:회식_운영규정#c01a002#01,57,회식_운영규정 제2조,"이 규정에서 사용하는 용어는 「법인카드 사용 규정」제2조를 따르며, 회식 유형은 다음과 같이 정의한다."
8,dump:회식_운영규정#c02a004#01,51,회식_운영규정 제4조,"회식은 다음 표에 따라 조직 단위별로 개최하며, 「직급체계」의 결재라인을 기준으로 승인한다."
9,dump:회식_운영규정#c02a005#02,98,회식_운영규정 제5조,"회식 1건이 2개 이상 가맹점(예: 식당 결제 후 별도 업소 결제)으로 나뉘는 경우, 후속 결제 건은 원칙적으로 회식비로 인정"



---
## 5. 큐 B — 표 (룰 임계값의 원천이라 전수 권장)

표가 깨지면 **룰엔진 임계값이 조용히 틀린다**(`policies/tiger_tables.py`). 자동 지표는
행 수·헤더 반복만 보므로 **값이 맞는지는 사람만 안다.**

**볼 것**: ⓐ 별표(한도표)의 금액·직책이 원문과 같은가 ⓑ 도입 문장이 붙어 무엇의 표인지 알 수 있는가
ⓒ 빈 격자 7건(파싱 결함)이 어떻게 나오는가.

In [9]:
# 세는 단위는 **표 요소**다 — 청크 기준으로 세면 부모가 표를 한 번 더 들고 있어 중복된다.
tbl_rows, tbl_chunk_ids = [], []
for name, doc in DOCS.items():
    owner = {eid: c for c in CHUNKS[name] for eid in c.element_ids}
    for e in doc.elements:
        if e.type != "table":
            continue
        c = owner.get(e.element_id)
        if c:
            tbl_chunk_ids.append(c.chunk_id)
        tbl_rows.append({
            "Document": name, "Element": e.element_id,
            "행(헤더포함)": e.attrs.get("rows") or 0,
            "Chunk ID": c.chunk_id if c else "⚠ 미소유",
            "Chunk Type": c.chunk_type if c else "", "Size": c.size if c else 0,
            "Citation": c.citation if c else "",
        })

tbl_df = pd.DataFrame(tbl_rows)
empty_ids = list(tbl_df.loc[tbl_df["행(헤더포함)"] == 0, "Chunk ID"])
print(f"표 요소(=논리 표) {len(tbl_df)}개 · 빈 격자 {len(empty_ids)}개 · "
      f"표를 담은 청크 {len(set(tbl_chunk_ids))}개")
display(tbl_df)

annex_chunks = [c for c in ALL_CHUNKS if c.chunk_type == "annex" and c.chunk_role != "parent"]
print(f"\n■ 별표 {len(annex_chunks)}건 — 한도표. **금액·직책이 원문과 같은지** 확인")
show_all(annex_chunks, limit=10)

표 요소(=논리 표) 39개 · 빈 격자 7개 · 표를 담은 청크 39개


,Document,Element,행(헤더포함),Chunk ID,Chunk Type,Size,Citation
0,법인카드_사용규정,법인카드_사용규정:p1:e3,3,dump:법인카드_사용규정#s002#01,table,72,법인카드_사용규정
1,법인카드_사용규정,법인카드_사용규정:p7:e112,6,dump:법인카드_사용규정#x001#01,annex,231,법인카드_사용규정
2,부서소개,부서소개:p2:e7,8,dump:부서소개#s003#01,table,802,부서소개
3,부서소개,부서소개:p3:e14,6,dump:부서소개#s005#01,table,561,부서소개
4,부서소개,부서소개:p3:e18,4,dump:부서소개#s007#01,table,310,부서소개
5,부서소개,부서소개:p4:e24,2,dump:부서소개#s008#01,table,173,부서소개
6,부서소개,부서소개:p4:e26,12,dump:부서소개#s009#01,table,624,부서소개
7,업무추진비_사용규정,업무추진비_사용규정:p6:e96,4,dump:업무추진비_사용규정#x001#01,annex,177,업무추진비_사용규정
8,업무추진비_사용규정,업무추진비_사용규정:p6:e99,5,dump:업무추진비_사용규정#x002#01,annex,212,업무추진비_사용규정
9,조직도,조직도:p2:e5,9,dump:조직도#s002#02,table,335,조직도



■ 별표 12건 — 한도표. **금액·직책이 원문과 같은지** 확인


… 2건 더 있음 (limit 인자를 올리세요)


In [10]:
# 빈 격자 표 — 파싱이 grid를 못 뽑은 것. 청킹 문제가 아니라 파싱 이관 대상(전략 §11 ⑦).
print("■ 빈 격자 표를 담은 청크")
show_all([BY_ID[i] for i in dict.fromkeys(empty_ids) if i in BY_ID], limit=8)

# 규정 4종의 표만 몰아 보기 (중복 제거)
reg = [BY_ID[i] for i in dict.fromkeys(tbl_chunk_ids)
       if i in BY_ID and DOCS[BY_ID[i].doc_name].profile == "REGULATION"]
print(f"■ 규정 문서의 표 청크 {len(reg)}건")
show_all(reg, limit=12)

■ 빈 격자 표를 담은 청크


■ 규정 문서의 표 청크 15건


… 3건 더 있음 (limit 인자를 올리세요)



---
## 6. 큐 C — 쪼개진 조 (잘린 자리 보기)

조가 예산을 넘겨 쪼개진 블록 89개다. **✂ 자리가 어색한가**를 본다 —
지표로는 `hard_split 0`(문장 중간 절단 없음)까지만 말할 수 있고, "항 중간에서 끊겨 말이 붕 뜨는가"는
읽어야 안다.

**볼 것**: 자식 청크가 `다만`·`이 경우` 같은 **지시어로 시작**하지 않는가(자동 판정 0건이지만 표본으로 확인),
부모 청크를 붙이면 복구되는가.

In [11]:
parents = [c for c in ALL_CHUNKS if c.chunk_role == "parent"]
split_rows = []
for p in parents:
    kids = [c for c in ALL_CHUNKS if c.parent_chunk_id == p.chunk_id]
    reason = ("예산 초과" if p.size > BUDGET.max else
              "표 분리" if any(k.has_table for k in kids) else "⚠ 이유 불명")
    split_rows.append({"Document": p.doc_name, "Parent": p.chunk_id, "Citation": p.citation,
                       "Parent Size": p.size, "Children": len(kids), "Reason": reason})
split_df = pd.DataFrame(split_rows)
display(split_df["Reason"].value_counts().rename("blocks").to_frame())
display(split_df.sort_values("Parent Size", ascending=False).head(20))

# ▼ 보고 싶은 조를 골라 실행 — 문서명·조 번호
article_view("회식_사용규정", 6)

,blocks
Reason,
예산 초과,67
표 분리,22


,Document,Parent,Citation,Parent Size,Children,Reason
78,부가가치세법,dump:부가가치세법#c06a068#P,부가가치세법 제60조,4644,6,예산 초과
57,법인세법,dump:법인세법#c04a158#P,법인세법 제93조,4350,44,예산 초과
83,여신전문금융업법,dump:여신전문금융업법#c01a002#P,여신전문금융업법 제2조,4227,46,예산 초과
56,법인세법,dump:법인세법#c04a157#P,법인세법 제93조,4180,41,예산 초과
62,법인세법,dump:법인세법#c04a168#P,법인세법 제98조,4063,4,예산 초과
39,법인세법,dump:법인세법#c01a077#P,법인세법 제55조의2,4021,3,예산 초과
61,법인세법,dump:법인세법#c04a167#P,법인세법 제98조,3347,4,예산 초과
46,법인세법,dump:법인세법#c01a112#P,법인세법 제75조의7,3159,3,예산 초과
24,법인세법,dump:법인세법#c01a030#P,법인세법 제24조,2803,3,예산 초과
27,법인세법,dump:법인세법#c01a036#P,법인세법 제29조,2579,3,예산 초과


■ 회식_운영규정 제6조 — 청크 1개 · 합계 788자


In [12]:
# 가장 많이 쪼개진 블록부터 순서대로 보기 (인덱스만 바꿔 가며 실행)
IDX = 0
row = split_df.sort_values("Children", ascending=False).iloc[IDX]
print(f"[{IDX}] {row['Citation']} — 자식 {row['Children']}개 / 부모 {row['Parent Size']}자")
article_view(row["Document"], BY_ID[row["Parent"]].article_no, limit=6, with_parent=True)

[0] 여신전문금융업법 제2조 — 자식 46개 / 부모 4227자
■ 여신전문금융업법 제2조 — 청크 46개 (+부모 1) · 합계 4137자


… 40건 더 있음 (limit 인자를 올리세요)


---
## 7. 큐 D — 인용 표기: **항인가 호인가** (§11 ④ 확정)

**결정이 끝났다.** 아래 7-1이 관례를 추측하는 대신 규정의 **자기인용 표기를 역산**해
답을 냈고(`제4조제2항의 부서 공용카드는 …`), 그 결과가 그대로 구현이 됐다.

| | 결정 전 | 결정 후 (`chunker._clause_kind`) |
|---|---|---|
| 판정 단위 | 프로파일 | **조(條)** |
| REGULATION | 일괄 `호` | 기본 `항` · 도입부에 `각 호`인 조만 `호` |
| LAW | `항` | `항` (변경 없음 — `clause_no`의 원천이 원문자 `①②③`이다) |

📏 실측 결과: 규정 4종의 조 27개 중 **26개가 `항`**, `호`는 **회식 제8조 하나뿐**.
바꾸기 전에는 그 26개 조가 전부 틀린 호칭으로 인용되고 있었다.

그래서 이 절은 이제 **판정하는 곳이 아니라 회귀를 눈으로 확인하는 곳**이다 —
아래 셀들의 `원문 근거` 열과 `현재 인용` 열이 전 행에서 같아야 한다.
(같은 계약을 `tests/test_chunking.py::test_clause_kind_follows_the_articles_own_wording`이 고정한다.)

> 인덱싱(임베딩) **후에** 바꿨다면 전량 재청킹 + 재임베딩이었다. upsert 전에 끝내 둔다.


In [13]:
rows = []
for name, doc in DOCS.items():
    if doc.profile != "REGULATION":
        continue
    for e in doc.elements:
        cno = e.attrs.get("clause_no")
        if cno is None:
            continue
        rows.append({
            "Document": name,
            "조": e.attrs.get("article_no"),
            "clause_no": cno,
            "원문 앞부분": e.text[:46].replace("\n", " ⏎ "),
        })
marker_df = pd.DataFrame(rows)
print(f"규정 문서의 clause 요소 {len(marker_df)}건 — 원문이 '1.'인가 '①'인가를 본다")
display(marker_df.head(30))

print("\n■ 지금 생성되는 인용 문자열")
for c in LEAVES:
    if DOCS[c.doc_name].profile == "REGULATION" and c.clause_start is not None:
        print(f"  {c.citation}")

규정 문서의 clause 요소 91건 — 원문이 '1.'인가 '①'인가를 본다


,Document,조,clause_no,원문 앞부분
0,법인카드_사용규정,4.0,1,1. 팀장 이상 직책(팀장·부서장·본부장·대표이사)을 보임한 임직원에게는 원칙적으로
1,법인카드_사용규정,4.0,2,"2. 직책이 없는 임직원(비직책자)은 부서 공용카드를 사용하거나, 업무상 필요가 인"
2,법인카드_사용규정,6.0,1,"1. 사용자는 법인카드를 선량한 관리자의 주의로 보관·사용하여야 하며, 타인에게 양"
3,법인카드_사용규정,6.0,2,"2. 퇴사, 휴직, 부서 이동 시 사용자는 지체 없이 법인카드를 반납하여야 한다."
4,법인카드_사용규정,9.0,1,1. 사적 용도의 지출
5,법인카드_사용규정,9.0,2,"2. 유흥업소, 사행성 업종에서의 지출"
6,법인카드_사용규정,9.0,3,3. 상품권 등 유가증권의 현금 구매(부득이 상품권을 구입하는 경우 반드시 신용카드
7,법인카드_사용규정,9.0,4,4. 업무 관련성이 객관적으로 소명되지 않는 지출
8,법인카드_사용규정,9.0,5,5. 개인 명의 카드로 결제 후 회사에 청구하는 방식의 우회 사용
9,법인카드_사용규정,10.0,1,1. 1일 사용한도 및 월 사용한도는 직책별로 별표 1과 같이 정한다.



■ 지금 생성되는 인용 문자열
  법인카드_사용규정 제4조 제1~2항
  법인카드_사용규정 제6조 제1~2항
  법인카드_사용규정 제9조 제1~5항
  법인카드_사용규정 제10조 제1~3항
  법인카드_사용규정 제11조 제1~4항
  법인카드_사용규정 제12조 제1~3항
  법인카드_사용규정 제13조 제1~2항
  법인카드_사용규정 제14조 제1~3항
  법인카드_사용규정 제17조 제1~2항
  업무추진비_사용규정 제2조 제1~3항
  업무추진비_사용규정 제5조 제1~3항
  업무추진비_사용규정 제6조 제1~5항
  업무추진비_사용규정 제6조 제6~7항
  업무추진비_사용규정 제7조 제1~3항
  업무추진비_사용규정 제11조 제1~2항
  업무추진비_사용규정 제12조 제1~3항
  업무추진비_사용규정 제13조 제1~2항
  출장비_사용규정 제4조 제1~3항
  출장비_사용규정 제5조 제1~2항
  출장비_사용규정 제7조 제1~2항
  출장비_사용규정 제8조 제1~3항
  출장비_사용규정 제9조 제1~3항
  출장비_사용규정 제10조 제1~2항
  출장비_사용규정 제11조 제1~3항
  출장비_사용규정 제12조 제1~3항
  회식_운영규정 제6조 제1~5항
  회식_운영규정 제8조 제1~8호
  회식_운영규정 제9조 제1~6항
  회식_운영규정 제1~2항



### 7-1. 원문이 자기 항목을 뭐라고 부르는가 — 관례 추측 대신 **역산**

호칭은 규정이 **스스로 인용할 때** 드러난다(`제4조제2항의 부서 공용카드는 …`).
아래 두 셀이 ⓐ 상호참조 표기를 세고 ⓑ 조마다 `다음 각 호` 도입부가 있는지로 **항/호를 판정**한다.

In [14]:
# ⓐ 규정이 스스로를 인용할 때 쓰는 표기
ref_rows = []
for name, doc in DOCS.items():
    if doc.profile != "REGULATION":
        continue
    txt = "\n".join(e.text for e in doc.body())
    ref_rows.append({"Document": name,
                     "제N항 참조": len(re.findall(r"제\s?\d+\s?항", txt)),
                     "제N호 참조": len(re.findall(r"제\s?\d+\s?호", txt))})
display(pd.DataFrame(ref_rows))

print("■ 참조 문맥 — 무엇을 가리키는지 직접 확인")
for name, doc in DOCS.items():
    if doc.profile != "REGULATION":
        continue
    for e in doc.body():
        for m in re.finditer(r".{0,26}제\s?\d+\s?[항호].{0,16}", e.text):
            print(f"  [{name}] …{m.group(0).strip()}…")

,Document,제N항 참조,제N호 참조
0,법인카드_사용규정,5,3
1,업무추진비_사용규정,8,5
2,출장비_사용규정,7,1
3,회식_운영규정,0,3


■ 참조 문맥 — 무엇을 가리키는지 직접 확인
  [법인카드_사용규정] …재상신·이의제기(어필)를 수행하는 자를 말한다(제2호의 "사용자"와 동 일인을 지…
  [법인카드_사용규정] …, 타인에게 양도·대여할 수 없다. 다만 제4조제2항의 부서 공용카드는 소속 부서…
  [법인카드_사용규정] …에 부서원이 순환하여 사용할 수 있으며, 이는 제1항의 양도·대여 금지 대상에 해…
  [법인카드_사용규정] …3. 부득이한 사유로 제1항·제2항의 사전승인 기준을 초과하는…
  [법인카드_사용규정] …교통비·숙박비·일비·식비 등 출장 관련 지출. 제1호부터 제4호 까지와는 별도의…
  [법인카드_사용규정] …※ 식대·기업업무추진비 지출은 제10조 제2항에 따라 직책과 무관하게 건당…
  [업무추진비_사용규정] …법인카드 운영 규정」과의 우선순위 명시(제6조제6항 신설), ② 청탁금지법 시행…
  [업무추진비_사용규정] …만원 수정, ③ 자기승인 방지 조항 신설(제6조제5항), ④ 승인 구간 실익 확보…
  [업무추진비_사용규정] …3. 제4조제4호(행사성 접대)에 해당하는 지…
  [업무추진비_사용규정] …5. (자기승인 방지) 제1항의 구간별 승인권자가 지출 담…
  [업무추진비_사용규정] …기업업무추진비로 분류되는 건은 동 규정 제8조제4호에 따라 지출 금액과 무관하게…
  [업무추진비_사용규정] …사전승인을 받아야 한다. 이 경우 제1항의 "30만원 이하 사후등록"…
  [업무추진비_사용규정] …하고, 30만원을 초과하는 구간의 승인권자는 제1항의 금액 구간별 기준을 그대로…
  [업무추진비_사용규정] …사전승인을 받지 아니하고 지출한 기업업무추진비(제2항의 사후승인 절차를 거친 경우…
  [업무추진비_사용규정] …접대 유형(제4조제1호부터 제5호까지 중 해당 유형)…
  [업무추진비_사용규정] …2. 제1항의 기록 중 참석자 명단 및…
  [업무추진비_사용규정] …3. 제1항 각 호의 기재가 누락되거나…
  [업무추진비_사용규정] …관계자의 경조사에 대한 화환·

In [15]:
# ⓑ 조별 판정 — 도입부에 "다음 각 호"가 있으면 호, 없으면 항
#    이 셀이 §11 ④의 근거였고, 지금은 구현(`chunker._clause_kind`)이 같은 판정을 하는지 보는 회귀다.
verdict = []
for name, doc in DOCS.items():
    if doc.profile != "REGULATION":
        continue
    arts: dict = {}
    for e in doc.body():
        a = e.attrs.get("article_no")
        if a is not None:
            arts.setdefault(a, []).append(e)
    for a, els in sorted(arts.items()):
        cl = [e for e in els if e.attrs.get("clause_no") is not None]
        if not cl:
            continue
        intro = " ".join(e.text for e in els if e.attrs.get("clause_no") is None)
        verdict.append({"Document": name, "조": f"제{int(a)}조", "항목": len(cl),
                        "원문 근거": "항" if not re.search(r"각\s?호", intro) else "호",
                        "현재 인용": next((c.clause_kind for c in CHUNKS[name]
                                        if c.article_no == a and c.clause_kind), "-"),
                        "도입부": intro[:44]})
vdf = pd.DataFrame(verdict)
display(vdf)
print(vdf.groupby(["원문 근거", "현재 인용"]).size().rename("조").to_frame())

mismatch = vdf[(vdf["현재 인용"] != "-") & (vdf["원문 근거"] != vdf["현재 인용"])]
if len(mismatch):
    print()
    print(f"❌ 원문과 어긋난 호칭 {len(mismatch)}건 — chunker._clause_kind 를 볼 것")
    display(mismatch)
else:
    print()
    print("✅ 원문 근거 = 현재 인용 (전 행) — §11 ④ 반영 확인")

,Document,조,항목,원문 근거,현재 인용,도입부
0,법인카드_사용규정,제4조,2,항,항,제4조 (발급 대상) 인을 받아 개인 카드를 발급받을 수 있다.
1,법인카드_사용규정,제6조,2,항,항,제6조 (관리 책임)
2,법인카드_사용규정,제9조,5,항,항,제9조 (사용 제한 및 금지 항목)
3,법인카드_사용규정,제10조,3,항,항,제10조 (사용 한도)
4,법인카드_사용규정,제11조,4,항,항,제11조 (증빙서류의 원칙) 거래처명 및 담당자 참석자 인원 및 소속 지출 목적
5,법인카드_사용규정,제12조,3,항,항,제12조 (정산 기한 및 절차)
6,법인카드_사용규정,제13조,2,항,항,제13조 (미정산·증빙 누락 시 조치)
7,법인카드_사용규정,제14조,3,항,항,"제14조 (기업업무추진비 한도 관리) 기본한도: 연 1,200만원 (회사는 설립"
8,법인카드_사용규정,제17조,2,항,항,제17조 (제재)
9,업무추진비_사용규정,제2조,3,항,항,제2조 (정의)


              조
원문 근거 현재 인용    
항     항      26
호     호       1

✅ 원문 근거 = 현재 인용 (전 행) — §11 ④ 반영 확인


In [16]:
# 반영 결과를 청크 하나로 눈에 넣는다 — 원문 마커와 인용을 같은 화면에서
sample = next((c for c in LEAVES
               if DOCS[c.doc_name].profile == "REGULATION" and c.clause_start is not None), None)
if sample:
    show(sample, note=f"인용은 '{sample.citation}' — 본문 마커와 맞는 호칭인가?")

# `호`로 판정된 유일한 조도 같이 본다 (회식 제8조 — 도입부에 `다음 각 호`)
ho = next((c for c in LEAVES if c.clause_kind == "호"), None)
if ho:
    show(ho, note=f"인용은 '{ho.citation}' — 도입부의 `각 호`가 근거다")
else:
    print("⚠️ `호`로 판정된 조가 없다 — 회식 제8조 판정이 깨졌는지 확인할 것")


---
## 8. 큐 E — 표본 40건 순회

평가 노트북이 버킷별로 할당 추출해 둔 40건(`review_samples.csv`)이다.
의심 청크만 보면 **"정상 청크는 어떤가"에 대한 감각이 안 생기므로** 이쪽도 봐야 한다.

`nxt()`를 반복 실행하면 한 건씩 넘어간다. 판정은 `mark(CUR.chunk_id, ...)`.

In [17]:
if SAMPLE_SHEET.exists():
    sheet = pd.read_csv(SAMPLE_SHEET, encoding="utf-8-sig")
    QUEUE = [(r["Bucket"], BY_ID[r["Chunk ID"]]) for _, r in sheet.iterrows()
             if r["Chunk ID"] in BY_ID]
    print(f"표본 {len(QUEUE)}건 로드 — {sheet['Bucket'].value_counts().to_dict()}")
else:   # 평가 노트북을 안 돌렸다면 즉석 표본
    import random
    random.seed(29)
    pool = [c for c in LEAVES if c.chunk_id not in REVIEW]
    QUEUE = [("adhoc", c) for c in random.sample(pool, min(40, len(pool)))]
    print(f"⚠ {SAMPLE_SHEET.name} 없음 → 즉석 표본 {len(QUEUE)}건")

POS = 0
CUR = None


def nxt(n: int = 1) -> None:
    """다음 표본으로. 이미 검수한 건은 건너뛴다."""
    global POS, CUR
    shown = 0
    while POS < len(QUEUE) and shown < n:
        bucket, c = QUEUE[POS]
        POS += 1
        if c.chunk_id in REVIEW and REVIEW[c.chunk_id].get("q1"):
            continue
        CUR = c
        print(f"[{POS}/{len(QUEUE)}] bucket={bucket}")
        show(c)
        print(f'mark("{c.chunk_id}", q1="Y", q2="Y", q3="N", memo="", queue="E")')
        shown += 1
    if POS >= len(QUEUE):
        print("표본 끝 — progress() 로 결과 확인")


nxt()

표본 40건 로드 — {'split_child': 10, 'flagged': 8, 'oversize': 6, 'table': 6, 'normal': 6, 'tiny': 4}
[1/40] bucket=flagged


mark("dump:여신전문금융업법#c02a021#01", q1="Y", q2="Y", q3="N", memo="", queue="E")


In [18]:
nxt()      # 이 셀을 반복 실행하며 넘긴다. 판정은 위에 찍힌 mark(...) 줄을 복사해 값만 바꿔 실행

[2/40] bucket=flagged


mark("dump:법인세법#c01a059#02", q1="Y", q2="Y", q3="N", memo="", queue="E")



---
## 9. 자유 탐색 — 질의 정답셋 30건 만들기

우선순위 3번(질의 정답셋)을 **여기서 시작할 수 있다.** 임베딩이 없으므로 문자열 검색이지만,
"이 질문의 답이 어느 조에 있는가"를 확인하는 데는 충분하다.
정답은 **청크 ID가 아니라 조문 ID**(`문서명 + 조 라벨`)로 적는다 — 청킹을 바꿔도 정답이 안 무너지게(전략 §10.4).

In [19]:
hits = find("사전승인", limit=3)          # 룰이 참조하는 임계값 질의 후보
# find("유흥", limit=3)
# find("참석자", limit=3, docs=["회식_사용규정"])

print("\n■ 정답 후보 (조문 ID로 적을 것)")
for c in hits[:10]:
    print(f"  {c.doc_name} {c.article_label or '-'}   ← {c.chunk_id}")

"사전승인" — 23건


… 20건 더 있음 (limit 인자를 올리세요)

■ 정답 후보 (조문 ID로 적을 것)
  법인카드_사용규정 -   ← dump:법인카드_사용규정#s002#02
  법인카드_사용규정 제10조   ← dump:법인카드_사용규정#c03a010#01
  법인카드_사용규정 -   ← dump:법인카드_사용규정#x001#01
  법인카드_사용규정 -   ← dump:법인카드_사용규정#x001#02
  업무추진비_사용규정 -   ← dump:업무추진비_사용규정#s002#01
  업무추진비_사용규정 제6조   ← dump:업무추진비_사용규정#c02a006#01
  업무추진비_사용규정 제6조   ← dump:업무추진비_사용규정#c02a006#02
  업무추진비_사용규정 -   ← dump:업무추진비_사용규정#x002#01
  출장비_사용규정 -   ← dump:출장비_사용규정#s002#02
  출장비_사용규정 제2조   ← dump:출장비_사용규정#c01a002#01


In [20]:
# 문서 하나를 처음부터 훑기 — 청크 목록 먼저 보고 필요한 것만 펼친다
DOC = "법인카드_사용규정"
display(pd.DataFrame([{
    "Chunk ID": c.chunk_id, "Type": f"{c.chunk_type}/{c.chunk_role}", "Size": c.size,
    "Citation": c.citation, "Text": c.text[:60].replace("\n", " ⏎ "),
} for c in CHUNKS[DOC]]))

# show("dump:법인카드_사용규정#c03a010#01")

,Chunk ID,Type,Size,Citation,Text
0,dump:법인카드_사용규정#s002#P,section/parent,387,법인카드_사용규정,| 제정일 | 2026. 7. 20. | ⏎ |---|---| ⏎ | 시행일 | 2026. 8. 1. | ⏎ | 소관부
1,dump:법인카드_사용규정#s002#01,table/child,72,법인카드_사용규정,| 제정일 | 2026. 7. 20. | ⏎ |---|---| ⏎ | 시행일 | 2026. 8. 1. | ⏎ | 소관부
2,dump:법인카드_사용규정#s002#02,section/child,313,법인카드_사용규정,"개정이력 v1.0 제정 (조문 정합성 검토 반영) / v1.1 개정: 제2조 관리자 정의 정비, 제10조 사"
3,dump:법인카드_사용규정#c01a001#01,article/atomic,152,법인카드_사용규정 제1조,"이 규정은 타이거 주식회사(이하 ""회사"")가 임직원에게 발급하는 법인카드의 발급, 관리, 사용 및 정산에 관"
4,dump:법인카드_사용규정#c01a002#01,article/atomic,560,법인카드_사용규정 제2조,"이 규정에서 사용하는 용어의 정의는 다음과 같다. ⏎ ""법인카드""란 회사 명의로 발급되어 임직원이 업무 목적의"
5,dump:법인카드_사용규정#c01a003#01,article/atomic,37,법인카드_사용규정 제3조,이 규정은 회사로부터 법인카드를 발급받은 모든 임직원에게 적용한다.
6,dump:법인카드_사용규정#c02a004#01,article/atomic,145,법인카드_사용규정 제4조 제1~2항,1. 팀장 이상 직책(팀장·부서장·본부장·대표이사)을 보임한 임직원에게는 원칙적으로 개인 법인카드를 발 급한
7,dump:법인카드_사용규정#c02a005#01,article/atomic,83,법인카드_사용규정 제5조,"사용자는 소속 부서장의 승인을 받아 「법인카드 발급 신청서」를 경영지원본부에 제출하며, 경영지원본부는 신청"
8,dump:법인카드_사용규정#c02a006#01,article/atomic,224,법인카드_사용규정 제6조 제1~2항,"1. 사용자는 법인카드를 선량한 관리자의 주의로 보관·사용하여야 하며, 타인에게 양도·대여할 수 없다. 다만"
9,dump:법인카드_사용규정#c02a007#01,article/atomic,85,법인카드_사용규정 제7조,"사용자는 법인카드의 분실 또는 도난을 인지한 즉시 카드사에 사용정지를 요청하고, 24시간 이내에 경영지 ⏎ 원본"



---
## 10. 저장 · 진행률

`save_review()`는 자주 눌러도 된다(덮어쓰기). 검수가 끝나면 아래 셀의 요약을
`llm_wiki/_context/chunking-strategy.md` §11 ⑥에 반영한다.

In [21]:
save_review()
progress()

저장 0건 → D:\project\SKN29-FINAL-1TEAM\docling_eval\output\chunking\review_filled.csv
검수 0건 · 큐별 {}
  Q1 뜻 안 통함 0 · Q2 인용 불일치 0 · Q3 어색한 절단 0



---
## 11. 이 노트북이 판단하지 못하는 것

1. **검색이 잘 되는지는 여전히 모른다.** 여기서 좋아 보이는 청크가 임베딩 공간에서도
   잘 잡힌다는 보장은 없다. 그건 임베딩 모델 + 질의 정답셋이 있어야 한다(우선순위 2·3번).
2. **원문 대조는 사람이 원본을 봐야 한다.** 표 값이 맞는지는 이 노트북이 아니라
   `tiger_inc/` 원본(또는 인쇄본)과 비교해야 확정된다.
3. **파싱 결함은 여기서 못 고친다.** 빈 격자 표 7건·거짓 인용 1건은 원인이 파싱이므로,
   확인되면 전략 §11 ⑦(파싱 이관)으로 넘긴다.